In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import data, util

def load_image(path=''):
    if not path.strip():
        return data.camera()
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img

def show(images, titles):
    fig, axes = plt.subplots(1, len(images), figsize=(4*len(images), 4), squeeze=False)
    for ax, im, title in zip(axes.ravel(), images, titles):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

img = load_image(input('Image path (Enter for sample): '))

In [ ]:
# 1
padded = np.pad(img,1,mode='edge')
lbp = np.zeros_like(img)
h,w = img.shape
for bit,(dy,dx) in enumerate([(-1,-1),(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1)]):
    neighbor = padded[1+dy:1+dy+h,1+dx:1+dx+w]
    lbp |= ((neighbor>=img).astype(np.uint8)<<bit)
show([img,lbp],['Original','Local binary pattern'])

In [ ]:
# 2
from scipy.signal import wiener
from skimage.metrics import peak_signal_noise_ratio
k = int(input('Window size (3 or 5): '))
assert k in (3,5)
for mode in ['gaussian','speckle']:
    noisy = np.round(util.random_noise(img,mode=mode,var=.01)*255).astype(np.uint8)
    with np.errstate(divide='ignore',invalid='ignore'):
        result = wiener(noisy.astype(float),(k,k))
    result = np.where(np.isfinite(result),result,noisy)
    filters = [np.clip(result,0,255).astype(np.uint8),cv2.medianBlur(noisy,k),cv2.GaussianBlur(noisy,(k,k),0),cv2.bilateralFilter(noisy,k,50,50)]
    names = ['Wiener','Median','Gaussian','Bilateral']
    show([img,noisy,*filters],['Original',mode,*names])
    for name,out in zip(names,filters):
        print(mode,name,'PSNR:',peak_signal_noise_ratio(img,out,data_range=255))